In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

## 1. Reproducibility

In [2]:
SEED = 42
np.random.seed(SEED)

def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    markers = ["pyproject.toml", ".git"]

    for candidate in [start, *start.parents]:
        if any((candidate / marker).exists() for marker in markers):
            return candidate

    return start

PROJECT_ROOT = find_project_root()
OUTPUT_DIR = PROJECT_ROOT / "reports" / "surveillance_make_model_extension"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("OUTPUT_DIR:", OUTPUT_DIR)

PROJECT_ROOT: C:\Users\elesh\Academics\UTS\36127InnovationLab\Development\AutoVision_Model_Development
OUTPUT_DIR: C:\Users\elesh\Academics\UTS\36127InnovationLab\Development\AutoVision_Model_Development\reports\surveillance_make_model_extension


## 2. Locating input files

In [3]:
WEB_SPLIT_CANDIDATES = [
    PROJECT_ROOT
    / "reports"
    / "final_dataset_preparation"
    / "final_make_model_split.csv",
]

CLASS_MAP_CANDIDATES = [
    PROJECT_ROOT
    / "reports"
    / "final_dataset_preparation"
    / "final_make_model_class_to_id.json",

    PROJECT_ROOT
    / "models"
    / "config"
    / "final_make_model_class_to_id.json",
]

SURV_INVENTORY_CANDIDATES = [
    PROJECT_ROOT
    / "data"
    / "processed"
    / "compcars_surveillance_inventory_v2.csv",

    PROJECT_ROOT
    / "data"
    / "processed"
    / "compcars_surveillance_inventory.csv",
]

COLOUR_SPLIT_CANDIDATES = [
    PROJECT_ROOT
    / "reports"
    / "final_dataset_preparation"
    / "final_colour_split.csv",
]


def first_existing(candidates, label):
    for path in candidates:
        if path.exists():
            print(f"{label}: {path}")
            return path

    raise FileNotFoundError(
        f"Could not find {label}. Checked:\n"
        + "\n".join(str(path) for path in candidates)
    )


WEB_SPLIT_PATH = first_existing(
    WEB_SPLIT_CANDIDATES,
    "Web make/model split",
)

CLASS_MAP_PATH = first_existing(
    CLASS_MAP_CANDIDATES,
    "Make/model class map",
)

SURV_INVENTORY_PATH = first_existing(
    SURV_INVENTORY_CANDIDATES,
    "Surveillance inventory",
)

COLOUR_SPLIT_PATH = first_existing(
    COLOUR_SPLIT_CANDIDATES,
    "Final colour split with official surveillance split",
)

Web make/model split: C:\Users\elesh\Academics\UTS\36127InnovationLab\Development\AutoVision_Model_Development\reports\final_dataset_preparation\final_make_model_split.csv
Make/model class map: C:\Users\elesh\Academics\UTS\36127InnovationLab\Development\AutoVision_Model_Development\reports\final_dataset_preparation\final_make_model_class_to_id.json
Surveillance inventory: C:\Users\elesh\Academics\UTS\36127InnovationLab\Development\AutoVision_Model_Development\data\processed\compcars_surveillance_inventory_v2.csv
Final colour split with official surveillance split: C:\Users\elesh\Academics\UTS\36127InnovationLab\Development\AutoVision_Model_Development\reports\final_dataset_preparation\final_colour_split.csv


## 3. Loading frozen web split, class map and surveillance inventory

In [4]:
web_df = pd.read_csv(WEB_SPLIT_PATH)
surv_df = pd.read_csv(SURV_INVENTORY_PATH)

with open(CLASS_MAP_PATH, "r", encoding="utf-8") as f:
    class_to_id = json.load(f)

print("Web rows:", len(web_df))
print("Existing make/model classes:", len(class_to_id))
print("Surveillance rows:", len(surv_df))

print("\nWeb columns:")
print(web_df.columns.tolist())

print("\nSurveillance columns:")
print(surv_df.columns.tolist())

Web rows: 122099
Existing make/model classes: 1102
Surveillance rows: 44481

Web columns:
['file_path', 'make', 'model_name', 'body_type', 'width', 'height', 'source_dataset', 'image_hash', 'model_id', 'parent_brand', 'make_model', 'source_file_path', 'relative_path', 'dataset_key', 'split', 'class_id']

Surveillance columns:
['file_path', 'surveillance_model_id', 'make', 'model', 'web_model_id', 'color', 'width', 'height', 'corrupted', 'image_hash', 'source_dataset', 'body_type', 'parent_brand']


## 4. Standardising make/model labels

In [5]:
def clean_text(value):
    if pd.isna(value):
        return None
    value = str(value).strip()
    return value if value else None

if "make_model" not in web_df.columns:
    if {"make", "model_name"}.issubset(web_df.columns):
        web_df["make_model"] = (
            web_df["make"].map(clean_text)
            + " || "
            + web_df["model_name"].map(clean_text)
        )
    else:
        raise KeyError("Could not construct web make_model labels.")

if not {"make", "model"}.issubset(surv_df.columns):
    raise KeyError("Surveillance inventory must contain 'make' and 'model' columns.")

surv_df["make"] = surv_df["make"].map(clean_text)
surv_df["model"] = surv_df["model"].map(clean_text)
surv_df["make_model"] = surv_df["make"] + " || " + surv_df["model"]
surv_df["class_id"] = surv_df["make_model"].map(class_to_id)

print(
    surv_df[["make", "model", "make_model", "class_id"]]
    .head(10)
    .to_string(index=False)
)

make        model           make_model  class_id
Benz Benz S Class Benz || Benz S Class     161.0
Benz Benz S Class Benz || Benz S Class     161.0
Benz Benz S Class Benz || Benz S Class     161.0
Benz Benz S Class Benz || Benz S Class     161.0
Benz Benz S Class Benz || Benz S Class     161.0
Benz Benz S Class Benz || Benz S Class     161.0
Benz Benz S Class Benz || Benz S Class     161.0
Benz Benz S Class Benz || Benz S Class     161.0
Benz Benz S Class Benz || Benz S Class     161.0
Benz Benz S Class Benz || Benz S Class     161.0


## 5. Reconstructing official surveillance train/test membership

In [7]:
colour_split_df = pd.read_csv(COLOUR_SPLIT_PATH)

print("Colour split rows:", len(colour_split_df))
print("Colour split columns:")
print(colour_split_df.columns.tolist())


if "official_split" not in colour_split_df.columns:
    raise KeyError(
        "final_colour_split.csv does not contain an 'official_split' column."
    )


def surveillance_path_key(value):
    value = str(value).replace("\\", "/").strip()

    if "/image/" in value:
        return value.split("/image/", 1)[1]

    if value.startswith("image/"):
        return value[len("image/"):]

    if "sv_data/" in value:
        value = value.split("sv_data/", 1)[1]

        if value.startswith("image/"):
            value = value[len("image/"):]

        return value

    return value.lstrip("./")

surv_path_col = (
    "relative_path"
    if "relative_path" in surv_df.columns
    else "file_path"
)

colour_path_col = (
    "relative_path"
    if "relative_path" in colour_split_df.columns
    else "file_path"
)


print("\nUsing surveillance path column:", surv_path_col)
print("Using colour split path column:", colour_path_col)


surv_df["official_path_key"] = (
    surv_df[surv_path_col]
    .map(surveillance_path_key)
)

colour_split_df["official_path_key"] = (
    colour_split_df[colour_path_col]
    .map(surveillance_path_key)
)


print("\nSurveillance keys:")
print(
    surv_df[
        [surv_path_col, "official_path_key"]
    ].head(3).to_string(index=False)
)

print("\nColour split keys:")
print(
    colour_split_df[
        [colour_path_col, "official_path_key"]
    ].head(3).to_string(index=False)
)


split_lookup = (
    colour_split_df[
        [
            "official_path_key",
            "official_split",
        ]
    ]
    .drop_duplicates("official_path_key")
)

if "official_split" in surv_df.columns:
    surv_df = surv_df.drop(
        columns=["official_split"]
    )


surv_df = surv_df.merge(
    split_lookup,
    on="official_path_key",
    how="left",
)

print("\nOfficial split mapping:")

print(
    surv_df["official_split"]
    .fillna("unmatched")
    .value_counts()
)


matched_count = int(
    surv_df["official_split"]
    .notna()
    .sum()
)

unmatched_count = int(
    surv_df["official_split"]
    .isna()
    .sum()
)


print(
    f"\nMatched official split: "
    f"{matched_count:,} / {len(surv_df):,}"
)

print(
    f"Unmatched: "
    f"{unmatched_count:,}"
)

Colour split rows: 44441
Colour split columns:
['file_path', 'surveillance_model_id', 'make', 'model', 'web_model_id', 'color', 'width', 'height', 'corrupted', 'image_hash', 'source_dataset', 'body_type', 'parent_brand', 'colour_label', 'source_file_path', 'relative_path', 'dataset_key', 'official_split', 'split', 'class_id']

Using surveillance path column: file_path
Using colour split path column: relative_path

Surveillance keys:
                                                                                                      file_path     official_path_key
/content/drive/MyDrive/capstone-vehicle-metadata/data/compcars_sv/extracted/sv_data/image/28/4122b6ab038366.jpg 28/4122b6ab038366.jpg
/content/drive/MyDrive/capstone-vehicle-metadata/data/compcars_sv/extracted/sv_data/image/28/6e64ab2ec01e74.jpg 28/6e64ab2ec01e74.jpg
/content/drive/MyDrive/capstone-vehicle-metadata/data/compcars_sv/extracted/sv_data/image/28/e986d17ccbd8c2.jpg 28/e986d17ccbd8c2.jpg

Colour split keys:
       

## 6. Keeping valid make/model rows

In [8]:
working = surv_df.copy()

if "corrupted" in working.columns:
    before = len(working)
    working = working[~working["corrupted"].fillna(False).astype(bool)].copy()
    print("Removed corrupted rows:", before - len(working))

working = working[
    working["make"].notna()
    & working["model"].notna()
    & working["official_split"].isin(["train", "test"])
].copy()

print("Usable surveillance rows before taxonomy matching:", len(working))

Removed corrupted rows: 0
Usable surveillance rows before taxonomy matching: 44441


## 7. Measuring overlap with the existing taxonomy

In [9]:
matched = working[working["class_id"].notna()].copy()
unmatched = working[working["class_id"].isna()].copy()

matched["class_id"] = matched["class_id"].astype(int)

existing_classes = set(class_to_id.keys())
matched_classes = set(matched["make_model"].unique())
unmatched_classes = set(unmatched["make_model"].unique())

summary = {
    "existing_web_classes": len(existing_classes),
    "surveillance_rows_usable": len(working),
    "surveillance_rows_matched": len(matched),
    "surveillance_rows_unmatched": len(unmatched),
    "surveillance_classes_matched": len(matched_classes),
    "surveillance_classes_unmatched": len(unmatched_classes),
    "existing_classes_with_surveillance_support": len(existing_classes & matched_classes),
}

summary["row_match_rate"] = (
    summary["surveillance_rows_matched"] / summary["surveillance_rows_usable"]
    if summary["surveillance_rows_usable"] else 0
)

summary["existing_class_coverage"] = (
    summary["existing_classes_with_surveillance_support"] / summary["existing_web_classes"]
    if summary["existing_web_classes"] else 0
)

print(
    f"Matched surveillance rows: {len(matched):,} / {len(working):,} "
    f"({summary['row_match_rate']:.1%})"
)
print(
    f"Existing classes receiving surveillance support: "
    f"{summary['existing_classes_with_surveillance_support']:,} / "
    f"{summary['existing_web_classes']:,} "
    f"({summary['existing_class_coverage']:.1%})"
)

print("\nTop unmatched surveillance labels:")
display(
    unmatched["make_model"]
    .value_counts()
    .head(30)
    .rename_axis("make_model")
    .reset_index(name="images")
)

Matched surveillance rows: 43,739 / 44,441 (98.4%)
Existing classes receiving surveillance support: 274 / 1,102 (24.9%)

Top unmatched surveillance labels:


,make_model,images
0,Volkswagen || GOL,288
1,Lufeng || Fshion,125
2,Chrysler || Grand Voager’,91
3,GAC || Shuaijian,80
4,Huanghai || Aolong,45
5,Huanghai || Ruitu,42
6,Opel || Zafira’,31


## 8. Inspecting official train/test coverage after taxonomy matching

In [10]:
split_summary = (
    matched.groupby("official_split")
    .agg(
        rows=("make_model", "size"),
        classes=("make_model", "nunique"),
    )
    .reset_index()
)

display(split_summary)

train_matched = matched[matched["official_split"] == "train"].copy()
test_matched = matched[matched["official_split"] == "test"].copy()

print("Matched official train rows:", len(train_matched))
print("Matched official test rows:", len(test_matched))
print("Matched train classes:", train_matched["make_model"].nunique())
print("Matched test classes:", test_matched["make_model"].nunique())

,official_split,rows,classes
0,test,13113,274
1,train,30626,274


Matched official train rows: 30626
Matched official test rows: 13113
Matched train classes: 274
Matched test classes: 274


## 9. Creating validation from official surveillance train only

In [11]:
rng = np.random.default_rng(SEED)

def stratified_train_val_split(df, val_fraction=0.10):
    train_parts = []
    val_parts = []

    for _, group in df.groupby("make_model", sort=True):
        group = group.copy()
        idx = np.arange(len(group))
        rng.shuffle(idx)
        group = group.iloc[idx].reset_index(drop=True)

        if len(group) < 2:
            train_parts.append(group)
            continue

        n_val = int(round(len(group) * val_fraction))
        n_val = max(1, n_val)
        n_val = min(n_val, len(group) - 1)

        val_parts.append(group.iloc[:n_val].copy())
        train_parts.append(group.iloc[n_val:].copy())

    train_out = pd.concat(train_parts, ignore_index=True)
    val_out = (
        pd.concat(val_parts, ignore_index=True)
        if val_parts
        else df.iloc[0:0].copy()
    )

    return train_out, val_out

surv_train, surv_val = stratified_train_val_split(
    train_matched,
    val_fraction=0.10,
)

surv_train["split"] = "train"
surv_val["split"] = "val"
test_matched["split"] = "test"

print("Surveillance training rows:", len(surv_train))
print("Surveillance validation rows:", len(surv_val))
print("Official surveillance test rows:", len(test_matched))

Surveillance training rows: 27560
Surveillance validation rows: 3066
Official surveillance test rows: 13113


## 10. Leakage checks

In [12]:
def path_set(df):
    return set(df["official_path_key"].astype(str))

train_paths = path_set(surv_train)
val_paths = path_set(surv_val)
test_paths = path_set(test_matched)

print("train ∩ val:", len(train_paths & val_paths))
print("train ∩ test:", len(train_paths & test_paths))
print("val ∩ test:", len(val_paths & test_paths))

assert not (train_paths & val_paths)
assert not (train_paths & test_paths)
assert not (val_paths & test_paths)

print("Leakage checks passed.")

train ∩ val: 0
train ∩ test: 0
val ∩ test: 0
Leakage checks passed.


## 11. Building portable surveillance manifests

In [13]:
def make_portable(df):
    out = df.copy()

    if "relative_path" in out.columns:
        out["relative_path"] = (
            out["relative_path"]
            .astype(str)
            .str.replace("\\", "/", regex=False)
        )
    else:
        out["relative_path"] = "image/" + out["official_path_key"].astype(str)

    out["dataset_key"] = "compcars_surveillance"
    return out

surv_train_out = make_portable(surv_train)
surv_val_out = make_portable(surv_val)
surv_test_out = make_portable(test_matched)

KEEP_COLS = [
    "relative_path",
    "dataset_key",
    "make",
    "model",
    "make_model",
    "class_id",
    "official_split",
    "split",
    "width",
    "height",
    "body_type",
    "color",
    "parent_brand",
    "image_hash",
    "surveillance_model_id",
    "web_model_id",
]

def keep_existing_cols(df):
    return df[[c for c in KEEP_COLS if c in df.columns]].copy()

surv_train_out = keep_existing_cols(surv_train_out)
surv_val_out = keep_existing_cols(surv_val_out)
surv_test_out = keep_existing_cols(surv_test_out)

display(surv_train_out.head())

,relative_path,dataset_key,make,model,make_model,class_id,official_split,split,width,height,body_type,color,parent_brand,image_hash,surveillance_model_id,web_model_id
0,image/174/1673ba71a400e8.jpg,compcars_surveillance,Acura,Acura ILX,Acura || Acura ILX,2,train,train,747,747,sedan,white,Acura,18dddf745961e06c760c5eb7e619ae96,174,1639
1,image/174/ef50018f813f8a.jpg,compcars_surveillance,Acura,Acura ILX,Acura || Acura ILX,2,train,train,750,752,sedan,white,Acura,d2972d212238f19bf864472e068be216,174,1639
2,image/174/a59db35f6cbfdb.jpg,compcars_surveillance,Acura,Acura ILX,Acura || Acura ILX,2,train,train,756,757,sedan,white,Acura,1a5c45e6a6b4570ad092d84dfc4b2fbf,174,1639
3,image/174/6adf0669e9c4cd.jpg,compcars_surveillance,Acura,Acura ILX,Acura || Acura ILX,2,train,train,740,737,sedan,white,Acura,94b2d0c66d48e8aec06024d73b51442e,174,1639
4,image/174/2ab9890f11d6ac.jpg,compcars_surveillance,Acura,Acura ILX,Acura || Acura ILX,2,train,train,739,714,sedan,white,Acura,22c8a3023dfca9685c2b6d2c3c715da8,174,1639


## 12. Building mixed web and surveillance training manifests

In [14]:
web_mixed = web_df.copy()

if "relative_path" not in web_mixed.columns:
    raise KeyError(
        "Frozen web split is expected to contain portable 'relative_path'."
    )

if "dataset_key" not in web_mixed.columns:
    web_mixed["dataset_key"] = "compcars_web"

if "class_id" not in web_mixed.columns:
    web_mixed["class_id"] = web_mixed["make_model"].map(class_to_id)

web_train = web_mixed[web_mixed["split"] == "train"].copy()
web_val = web_mixed[web_mixed["split"] == "val"].copy()
web_test = web_mixed[web_mixed["split"] == "test"].copy()

mixed_train = pd.concat(
    [
        web_train.assign(training_source="compcars_web"),
        surv_train_out.assign(training_source="compcars_surveillance"),
    ],
    ignore_index=True,
    sort=False,
)

mixed_val = pd.concat(
    [
        web_val.assign(validation_domain="compcars_web"),
        surv_val_out.assign(validation_domain="compcars_surveillance"),
    ],
    ignore_index=True,
    sort=False,
)

print("Mixed training rows:", len(mixed_train))
print("  web train:", len(web_train))
print("  surveillance train:", len(surv_train_out))

print("\nMixed validation rows:", len(mixed_val))
print("  web val:", len(web_val))
print("  surveillance val:", len(surv_val_out))

print("\nTests remain separate:")
print("  frozen web test:", len(web_test))
print("  official surveillance test:", len(surv_test_out))

Mixed training rows: 125201
  web train: 97641
  surveillance train: 27560

Mixed validation rows: 15295
  web val: 12229
  surveillance val: 3066

Tests remain separate:
  frozen web test: 12229
  official surveillance test: 13113


## 13. Comparing class distributions

In [15]:
web_counts = web_train["make_model"].value_counts().rename("web_train")
surv_counts = surv_train_out["make_model"].value_counts().rename("surveillance_train")

class_comparison = (
    pd.concat([web_counts, surv_counts], axis=1)
    .fillna(0)
    .astype(int)
)

class_comparison["combined_train"] = (
    class_comparison["web_train"] + class_comparison["surveillance_train"]
)

class_comparison = class_comparison.reset_index()

display(
    class_comparison.sort_values(
        ["surveillance_train", "combined_train"],
        ascending=False,
    ).head(30)
)

,make_model,web_train,surveillance_train,combined_train
675,Volkswagen || Jetta,94,509,603
330,Jianghuai || Heyue,96,446,542
41,Nissan || Teana,123,369,492
672,Toyota || Huaguan,94,344,438
14,Toyota || Camry,156,320,476
115,Volkswagen || Sagitar,101,308,409
59,BMW || BMW X3,115,307,422
68,LAND-ROVER || Discovery,112,297,409
611,Skoda || Haorui,95,272,367
56,Volkswagen || Bora,117,265,382


## 14. Saving prepared files

In [16]:
paths = {
    "surveillance_train": OUTPUT_DIR / "surveillance_make_model_train.csv",
    "surveillance_val": OUTPUT_DIR / "surveillance_make_model_val.csv",
    "surveillance_official_test": OUTPUT_DIR / "surveillance_make_model_official_test.csv",
    "surveillance_unmatched": OUTPUT_DIR / "surveillance_make_model_unmatched.csv",
    "mixed_train": OUTPUT_DIR / "mixed_web_surveillance_train.csv",
    "mixed_val": OUTPUT_DIR / "mixed_web_surveillance_val.csv",
    "class_comparison": OUTPUT_DIR / "mixed_make_model_class_counts.csv",
    "summary": OUTPUT_DIR / "surveillance_extension_summary.json",
}

surv_train_out.to_csv(paths["surveillance_train"], index=False)
surv_val_out.to_csv(paths["surveillance_val"], index=False)
surv_test_out.to_csv(paths["surveillance_official_test"], index=False)

unmatched_save_cols = [
    c for c in [
        "relative_path",
        "make",
        "model",
        "make_model",
        "official_split",
        "surveillance_model_id",
        "web_model_id",
    ]
    if c in unmatched.columns
]
unmatched[unmatched_save_cols].to_csv(
    paths["surveillance_unmatched"],
    index=False,
)

mixed_train.to_csv(paths["mixed_train"], index=False)
mixed_val.to_csv(paths["mixed_val"], index=False)
class_comparison.to_csv(paths["class_comparison"], index=False)

summary.update(
    {
        "surveillance_train_rows": len(surv_train_out),
        "surveillance_val_rows": len(surv_val_out),
        "surveillance_official_test_rows": len(surv_test_out),
        "mixed_train_rows": len(mixed_train),
        "mixed_val_rows": len(mixed_val),
        "frozen_web_test_rows": len(web_test),
        "seed": SEED,
    }
)

with open(paths["summary"], "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=4)

for name, path in paths.items():
    print(f"{name}: {path}")

surveillance_train: C:\Users\elesh\Academics\UTS\36127InnovationLab\Development\AutoVision_Model_Development\reports\surveillance_make_model_extension\surveillance_make_model_train.csv
surveillance_val: C:\Users\elesh\Academics\UTS\36127InnovationLab\Development\AutoVision_Model_Development\reports\surveillance_make_model_extension\surveillance_make_model_val.csv
surveillance_official_test: C:\Users\elesh\Academics\UTS\36127InnovationLab\Development\AutoVision_Model_Development\reports\surveillance_make_model_extension\surveillance_make_model_official_test.csv
surveillance_unmatched: C:\Users\elesh\Academics\UTS\36127InnovationLab\Development\AutoVision_Model_Development\reports\surveillance_make_model_extension\surveillance_make_model_unmatched.csv
mixed_train: C:\Users\elesh\Academics\UTS\36127InnovationLab\Development\AutoVision_Model_Development\reports\surveillance_make_model_extension\mixed_web_surveillance_train.csv
mixed_val: C:\Users\elesh\Academics\UTS\36127InnovationLab\Deve

## 15. Final integrity checks

In [17]:
def count_windows_absolute(series):
    s = series.astype(str)
    return int(s.str.match(r"^[A-Za-z]:\\").sum())

print(
    "Surveillance train absolute Windows paths:",
    count_windows_absolute(surv_train_out["relative_path"]),
)
print(
    "Surveillance val absolute Windows paths:",
    count_windows_absolute(surv_val_out["relative_path"]),
)
print(
    "Surveillance test absolute Windows paths:",
    count_windows_absolute(surv_test_out["relative_path"]),
)

assert count_windows_absolute(surv_train_out["relative_path"]) == 0
assert count_windows_absolute(surv_val_out["relative_path"]) == 0
assert count_windows_absolute(surv_test_out["relative_path"]) == 0

assert not (
    set(surv_test_out["relative_path"])
    & set(surv_train_out["relative_path"])
)
assert not (
    set(surv_test_out["relative_path"])
    & set(surv_val_out["relative_path"])
)

print("Final integrity checks passed.")

Surveillance train absolute Windows paths: 0
Surveillance val absolute Windows paths: 0
Surveillance test absolute Windows paths: 0
Final integrity checks passed.
